# SIMPACT 2026: Foundation Model Fine-Tuning
This notebook downloads your How2Sign (H2S) Foundation Model directly from Weights & Biases (wandb), runs a pipeline test, and fine-tunes the network on your new medical phrases.

In [ ]:
# --- 1. DOWNLOAD MODEL FROM WEIGHTS & BIASES ---
import wandb
import os

# TODO: Adjust this path to your exact wandb username/project/artifact
WANDB_ARTIFACT_PATH = "your_username/your_project/h2s_model:latest"

print(f"Connecting to wandb to download foundation model: {WANDB_ARTIFACT_PATH}")
try:
    api = wandb.Api()
    artifact = api.artifact(WANDB_ARTIFACT_PATH)
    download_dir = artifact.download(root="./h2s_foundation_model")
    
    # Find the .pt file in the downloaded folder
    pt_files = [f for f in os.listdir(download_dir) if f.endswith('.pt')]
    if pt_files:
        FOUNDATION_MODEL_PATH = os.path.join(download_dir, pt_files[0])
        print(f"\nSuccess! Downloaded Foundation Model: {FOUNDATION_MODEL_PATH}")
    else:
        print("\nError: Downloaded artifact, but no .pt file found inside.")
        FOUNDATION_MODEL_PATH = "model_not_found.pt"
except Exception as e:
    print(f"\nFailed to download from wandb. Error: {e}")
    print("Make sure you are logged in (run 'wandb login' in terminal) and the artifact path is correct.")
    FOUNDATION_MODEL_PATH = "sana_psl_complete_55mb.pt" # Fallback

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# --- 2. CONFIGURATION ---
CONFIG = {
    "DATASET_DIR":      "E:/sign-language/dataset",
    "PRETRAINED_MODEL": FOUNDATION_MODEL_PATH,  # Dynamically points to downloaded W&B file
    "MT5_MODEL_NAME":   "google/mt5-small",
    "D_MODEL":          512,
    "NUM_HEADS":        8,
    "NUM_ENCODER_LAYERS": 2,
    "DIM_FEEDFORWARD":  1024,
    "DROPOUT":          0.108,
    "MAX_SEQ_LEN":      100,
    "MAX_TARGET_LEN":   32,
    "INPUT_DIM":        208,
    "EPOCHS":           60,     # High epochs for 40-video overfitting test
    "BATCH_SIZE":       4,
    "LEARNING_RATE":    5e-5,
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Defining the Architecture (Exactly as Trained on H2S)

In [ ]:
# --- 3. SANA ARCHITECTURE ---
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class TemporalGestureTokenizer(nn.Module):
    def __init__(self, input_dim=208, d_model=512):
        super().__init__()
        self.conv1 = nn.Conv1d(input_dim, d_model // 2, kernel_size=5, stride=2, padding=2)
        self.norm1 = nn.BatchNorm1d(d_model // 2)
        self.gelu  = nn.GELU()
        self.conv2 = nn.Conv1d(d_model // 2, d_model, kernel_size=5, stride=2, padding=2)
        self.norm2 = nn.BatchNorm1d(d_model)
        
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.norm1(x)
        x = self.gelu(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = self.gelu(x)
        x = x.transpose(1, 2)
        return x

class UpgradedSpatialTemporalEncoder(nn.Module):
    def __init__(self, input_dim, d_model, num_heads, num_layers, ffn_dim, dropout, max_len):
        super().__init__()
        self.tokenizer = TemporalGestureTokenizer(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model, max_len=100)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ffn_dim,
            dropout=dropout, activation="gelu", batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        
    def forward(self, src):
        x = self.tokenizer(src)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = self.norm(x)
        return x

class SANA_PSL_Translator(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.visual_encoder = UpgradedSpatialTemporalEncoder(
            input_dim=config["INPUT_DIM"],
            d_model=config["D_MODEL"],
            num_heads=config["NUM_HEADS"],
            num_layers=config["NUM_ENCODER_LAYERS"],
            ffn_dim=config["DIM_FEEDFORWARD"],
            dropout=config["DROPOUT"],
            max_len=config["MAX_SEQ_LEN"]
        )
        self.mt5 = AutoModelForSeq2SeqLM.from_pretrained(config["MT5_MODEL_NAME"])
        
    def forward(self, src, labels=None):
        encoder_outputs = self.visual_encoder(src)
        if labels is not None:
            outputs = self.mt5(encoder_outputs= (encoder_outputs,), labels=labels)
        else:
            outputs = self.mt5(encoder_outputs= (encoder_outputs,))
        return outputs


In [ ]:
# --- 4. LOADING THE H2S FOUNDATION MODEL & TOKENIZER ---
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MT5_MODEL_NAME"])
model = SANA_PSL_Translator(CONFIG)

if os.path.exists(CONFIG["PRETRAINED_MODEL"]):
    print(f"Loading existing Foundation weights from {CONFIG['PRETRAINED_MODEL']}...")
    # Handle state dict structure depending on how you saved it in wandb
    checkpoint = torch.load(CONFIG["PRETRAINED_MODEL"], map_location=device)
    if "model_state_dict" in checkpoint:
        # If you saved only the visual encoder previously, this handles it
        try:
            model.load_state_dict(checkpoint["model_state_dict"])
        except:
            model.visual_encoder.load_state_dict(checkpoint["model_state_dict"])
    else:
        model.load_state_dict(checkpoint)
    print("Successfully loaded Foundation Model!")
else:
    print("WARNING: Pretrained model not found! Training from scratch.")
model = model.to(device)


### The Medical Dataset Loader

In [ ]:
# --- 5. DATASET PREPARATION ---
class MedicalDataset(Dataset):
    def __init__(self, dataset_dir, tokenizer, config):
        self.samples = []
        self.tokenizer = tokenizer
        self.config = config
        
        if not os.path.exists(dataset_dir):
            print(f"Dataset dir {dataset_dir} not found.")
            return
            
        for folder in os.listdir(dataset_dir):
            folder_path = os.path.join(dataset_dir, folder)
            if os.path.isdir(folder_path):
                label = folder.replace("_", " ")
                for file in os.listdir(folder_path):
                    if file.endswith(".npy"):
                        self.samples.append((os.path.join(folder_path, file), label))
        print(f"Found {len(self.samples)} samples across {len(os.listdir(dataset_dir))} classes.")
        
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        npy_path, label = self.samples[idx]
        data = np.load(npy_path)
        
        # Sequence Padding/Truncating
        T = data.shape[0]
        if T > self.config["MAX_SEQ_LEN"]:
            data = data[:self.config["MAX_SEQ_LEN"]]
        elif T < self.config["MAX_SEQ_LEN"]:
            padding = np.zeros((self.config["MAX_SEQ_LEN"] - T, self.config["INPUT_DIM"]))
            data = np.vstack([data, padding])
            
        data_tensor = torch.tensor(data, dtype=torch.float32)
        
        tokenized = self.tokenizer(
            label,
            padding="max_length",
            truncation=True,
            max_length=self.config["MAX_TARGET_LEN"],
            return_tensors="pt"
        )
        label_tensor = tokenized.input_ids.squeeze(0)
        label_tensor[label_tensor == self.tokenizer.pad_token_id] = -100
        
        return data_tensor, label_tensor, label

med_dataset = MedicalDataset(CONFIG["DATASET_DIR"], tokenizer, CONFIG)
med_loader = DataLoader(med_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)

### Step 1: INFERENCE TEST (Verify Pipeline works on H2S Weights)


In [ ]:
if len(med_dataset) > 0:
    test_data, _, true_label = med_dataset[0]
    test_data = test_data.unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        encoder_outputs = model.visual_encoder(test_data)
        outputs = model.mt5.generate(
            encoder_outputs=(encoder_outputs,),
            max_length=CONFIG["MAX_TARGET_LEN"]
        )
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
    print(f"\n--- PIPELINE INFERENCE TEST ---")
    print(f"Ground Truth (What you signed): {true_label}")
    print(f"H2S Foundation Model Raw Prediction: {pred_text}")
    print(f"Pipeline status: SUCCESS! (Output text is expected to be wrong before fine-tuning)")

### Step 2: TRANSFER LEARNING (Fine-Tuning on Medical Videos)

In [ ]:
# Freeze mT5, unfreeze cross-attention to train fast
for param in model.mt5.parameters():
    param.requires_grad = False
for block in model.mt5.decoder.block:
    for param in block.layer[1].parameters():
        param.requires_grad = True
        
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG["LEARNING_RATE"])

print("Starting Fine-Tuning on the Medical Dataset...")
for epoch in range(CONFIG["EPOCHS"]):
    model.train()
    total_loss = 0
    
    for batch_data, batch_labels, _ in med_loader:
        batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(src=batch_data, labels=batch_labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{CONFIG['EPOCHS']}] | Loss: {total_loss/len(med_loader):.4f}")

torch.save(model.state_dict(), "sana_psl_medical_finetuned.pt")
print("Fine-tuning complete! Saved weights as sana_psl_medical_finetuned.pt")